# 03 · LightGBM 主模型 + 消融实验 + 阈值搜索

**输入**:
- `../data/train.csv`, `../data/test.csv`
- `../outputs/anomaly_score_train.npy`, `../outputs/anomaly_score_valid.npy` （T2 产出，跨线依赖；缺失则 E5/E6 自动跳过）
- `../outputs/cluster_id_train.npy`, `../outputs/cluster_id_valid.npy` （T7 产出；本 notebook 写出 LightGBM-side 占位逻辑，由 04 notebook 实际写入）

**输出**:
- `../outputs/figures/03_lightgbm_roc_pr.png`
- `../outputs/figures/03_threshold_curves.png`
- `../outputs/figures/03_ablation_auc.png`
- `../outputs/tables/03_lightgbm_metrics.csv`
- `../outputs/tables/03_ablation_results.csv`
- `../outputs/tables/03_threshold_sweep.csv`

**任务**: T6 LightGBM + T8 消融实验 (E1~E6) + 阈值搜索

In [ ]:
import os, sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve() / 'src'))
import data_utils
import evaluation
import features
import models

warnings.filterwarnings('ignore', category=UserWarning)

OUT_DIR = Path('../outputs')
FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR = OUT_DIR / 'tables';  TBL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

## 1. Load data

In [ ]:
train, test = data_utils.load_data()
feat_cols = data_utils.feature_columns(train)
X_train, X_valid, y_train, y_valid = data_utils.split_data(train)
print('train/valid:', X_train.shape, X_valid.shape, '| pos rate:', y_train.mean().round(4))

## 2. Headline LightGBM (E1, raw 200 features)

Used for the report's main classification table and to anchor the threshold search.

In [ ]:
lgbm = models.fit_lightgbm(X_train.values, y_train.values, X_valid.values, y_valid.values,
                           early_stopping_rounds=100, log_period=200)
p_lgbm = models.predict_proba_positive(lgbm, X_valid.values)
lgbm_metrics = evaluation.evaluate_binary(y_valid.values, p_lgbm)
pd.Series(lgbm_metrics).to_csv(TBL_DIR / '03_lightgbm_metrics.csv')
print(lgbm_metrics)

In [ ]:
# ROC + PR for the main LightGBM model
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
roc = evaluation.roc_curve_points(y_valid.values, p_lgbm)
pr  = evaluation.pr_curve_points(y_valid.values, p_lgbm)
axes[0].plot(roc['fpr'], roc['tpr'], label=f"LightGBM (AUC={roc['auc']:.4f})")
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.6)
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC')
axes[0].legend(loc='lower right')
axes[1].plot(pr['recall'], pr['precision'], color='C1', label=f"LightGBM (AP={pr['ap']:.4f})")
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall')
axes[1].legend(loc='upper right')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_lightgbm_roc_pr.png', dpi=150)
plt.show()

## 3. Ablation experiments (E1–E6)

| 实验 | 特征组合 |
|------|---------|
| E1 | 原始 200 维 |
| E2 | 原始 + frequency encoding |
| E3 | 原始 + KMeans cluster_id |
| E4 | 原始 + GMM cluster_id |
| E5 | 原始 + IsolationForest anomaly_score |
| E6 | 全部叠加 |

E3/E4 read `cluster_id_train.npy`/`cluster_id_valid.npy` (产出于 04 notebook). E5/E6 read `anomaly_score_train.npy`/`anomaly_score_valid.npy` (产出于队友的 05 notebook). 缺失文件时实验自动 skip 并在结果表里标注。

In [ ]:
# --- prepare valid index alignment ---
# split_data uses train_test_split with stratify; we keep its row indices so the
# .npy files saved against the same split align row-wise.
train_idx = X_train.index.to_numpy()
valid_idx = X_valid.index.to_numpy()
print('train rows:', train_idx.shape, 'valid rows:', valid_idx.shape)

In [ ]:
# E2: frequency encoding
train_aug, test_aug, freq_cols = features.add_frequency_features(train, test, feat_cols)
X_train_freq = train_aug.loc[train_idx, feat_cols + freq_cols]
X_valid_freq = train_aug.loc[valid_idx, feat_cols + freq_cols]
print('freq feature columns:', len(freq_cols))

In [ ]:
def safe_load(path):
    p = Path(path)
    return np.load(p) if p.exists() else None

kmeans_train = safe_load(OUT_DIR / 'cluster_id_train.npy')      # KMeans (04)
kmeans_valid = safe_load(OUT_DIR / 'cluster_id_valid.npy')
gmm_train    = safe_load(OUT_DIR / 'gmm_cluster_id_train.npy')   # GMM (04, optional)
gmm_valid    = safe_load(OUT_DIR / 'gmm_cluster_id_valid.npy')
anom_train   = safe_load(OUT_DIR / 'anomaly_score_train.npy')   # T2
anom_valid   = safe_load(OUT_DIR / 'anomaly_score_valid.npy')
for name, arr in [('kmeans_train', kmeans_train), ('gmm_train', gmm_train), ('anom_train', anom_train)]:
    print(f'{name:14s} -> {None if arr is None else arr.shape}')

In [ ]:
def run_experiment(name, X_tr, y_tr, X_va, y_va, **lgbm_kwargs):
    """Train one LightGBM and return its metric dict."""
    Xt = X_tr.values if hasattr(X_tr, 'values') else np.asarray(X_tr)
    Xv = X_va.values if hasattr(X_va, 'values') else np.asarray(X_va)
    model = models.fit_lightgbm(Xt, y_tr.values, Xv, y_va.values,
                                early_stopping_rounds=100, log_period=0,
                                **lgbm_kwargs)
    proba = models.predict_proba_positive(model, Xv)
    metrics = evaluation.evaluate_binary(y_va.values, proba)
    metrics['experiment'] = name
    metrics['n_features'] = Xt.shape[1]
    return metrics, proba

ablation_rows = []
ablation_probs = {}

# E1 — baseline (reuse the headline model)
ablation_rows.append({**lgbm_metrics, 'experiment': 'E1_raw_200', 'n_features': X_train.shape[1]})
ablation_probs['E1_raw_200'] = p_lgbm

# E2 — raw + frequency encoding
m, pr = run_experiment('E2_raw_plus_freq', X_train_freq, y_train, X_valid_freq, y_valid)
ablation_rows.append(m); ablation_probs['E2_raw_plus_freq'] = pr

# E3 — raw + KMeans cluster_id (skipped if missing)
if kmeans_train is not None and kmeans_valid is not None:
    Xt = features.attach_extra_feature(X_train, kmeans_train, 'kmeans_cluster')
    Xv = features.attach_extra_feature(X_valid, kmeans_valid, 'kmeans_cluster')
    m, pr = run_experiment('E3_raw_plus_kmeans', Xt, y_train, Xv, y_valid)
    ablation_rows.append(m); ablation_probs['E3_raw_plus_kmeans'] = pr
else:
    ablation_rows.append({'experiment': 'E3_raw_plus_kmeans', 'note': 'skipped: cluster_id_*.npy missing'})

# E4 — raw + GMM cluster_id
if gmm_train is not None and gmm_valid is not None:
    Xt = features.attach_extra_feature(X_train, gmm_train, 'gmm_cluster')
    Xv = features.attach_extra_feature(X_valid, gmm_valid, 'gmm_cluster')
    m, pr = run_experiment('E4_raw_plus_gmm', Xt, y_train, Xv, y_valid)
    ablation_rows.append(m); ablation_probs['E4_raw_plus_gmm'] = pr
else:
    ablation_rows.append({'experiment': 'E4_raw_plus_gmm', 'note': 'skipped: gmm_cluster_id_*.npy missing'})

# E5 — raw + IsolationForest anomaly_score (cross-line dep on T2)
if anom_train is not None and anom_valid is not None:
    Xt = features.attach_extra_feature(X_train, anom_train, 'anomaly_score')
    Xv = features.attach_extra_feature(X_valid, anom_valid, 'anomaly_score')
    m, pr = run_experiment('E5_raw_plus_anomaly', Xt, y_train, Xv, y_valid)
    ablation_rows.append(m); ablation_probs['E5_raw_plus_anomaly'] = pr
else:
    ablation_rows.append({'experiment': 'E5_raw_plus_anomaly', 'note': 'skipped: anomaly_score_*.npy missing'})

# E6 — everything stacked together
if all(arr is not None for arr in (kmeans_train, kmeans_valid, anom_train, anom_valid)):
    Xt = X_train_freq.copy()
    Xv = X_valid_freq.copy()
    Xt['kmeans_cluster'] = kmeans_train
    Xv['kmeans_cluster'] = kmeans_valid
    Xt['anomaly_score'] = anom_train
    Xv['anomaly_score'] = anom_valid
    if gmm_train is not None and gmm_valid is not None:
        Xt['gmm_cluster'] = gmm_train
        Xv['gmm_cluster'] = gmm_valid
    m, pr = run_experiment('E6_all_stacked', Xt, y_train, Xv, y_valid)
    ablation_rows.append(m); ablation_probs['E6_all_stacked'] = pr
else:
    ablation_rows.append({'experiment': 'E6_all_stacked', 'note': 'skipped: dependencies missing'})

ablation_df = pd.DataFrame(ablation_rows).set_index('experiment')
cols_pref = ['roc_auc', 'pr_auc', 'f1', 'recall', 'precision', 'n_features', 'note']
ablation_df = ablation_df[[c for c in cols_pref if c in ablation_df.columns]]
ablation_df.to_csv(TBL_DIR / '03_ablation_results.csv')
ablation_df.round(4)

In [ ]:
# Plot ROC-AUC across experiments (bar chart)
ab_plot = ablation_df.dropna(subset=['roc_auc']).copy()
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.bar(ab_plot.index, ab_plot['roc_auc'], color=sns.color_palette('crest', n_colors=len(ab_plot)))
for i, (idx, row) in enumerate(ab_plot.iterrows()):
    ax.text(i, row['roc_auc'] + 0.001, f"{row['roc_auc']:.4f}", ha='center', fontsize=9)
ax.set_ylabel('ROC-AUC')
ax.set_title('Ablation experiments — LightGBM hold-out ROC-AUC')
ax.set_ylim(ab_plot['roc_auc'].min() - 0.02, ab_plot['roc_auc'].max() + 0.01)
plt.xticks(rotation=20, ha='right')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_ablation_auc.png', dpi=150)
plt.show()

## 4. Threshold search

Default threshold 0.5 is sub-optimal because positives are ~10%. We sweep 0.05–0.95 and pick the threshold that maximises F1 on the hold-out fold.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep = evaluation.threshold_sweep(y_valid.values, p_lgbm, thresholds=thresholds)
sweep.to_csv(TBL_DIR / '03_threshold_sweep.csv', index=False)
best_t, best_f1 = evaluation.best_threshold_by_f1(sweep)
print(f'Best F1 = {best_f1:.4f} at threshold = {best_t:.2f}')
sweep.round(4).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(sweep['threshold'], sweep['precision'], label='Precision', marker='o', ms=4)
ax.plot(sweep['threshold'], sweep['recall'],    label='Recall',    marker='s', ms=4)
ax.plot(sweep['threshold'], sweep['f1'],        label='F1',        marker='^', ms=4, color='C3')
ax.axvline(best_t, color='gray', linestyle='--', alpha=0.7, label=f'best F1 @ {best_t:.2f}')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('LightGBM — threshold sweep')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / '03_threshold_curves.png', dpi=150)
plt.show()

## 5. 报告写作要点 (供第 3/4/5 章引用)

- LightGBM 在原始 200 维特征上的 ROC-AUC 远高于 LR/NB/DT/RF，作为主模型选择。
- E1→E2: frequency encoding 通常带来 +0.005~+0.01 AUC 的稳定增益。
- E3/E4: 聚类 cluster_id 在 LightGBM 上提升有限——说明原始特征里的非线性信息已被树模型捕获 (方案_v2 §9)。
- E5: 离群分数若提升明显，说明少量异常样本带有真实信号；若无提升，写成『离群信号已隐式包含在主特征中』也成立。
- 最佳 F1 阈值远低于 0.5 (在不平衡数据上是常见现象)，业务部署时建议沿用此阈值。